# Package a vLLM model with Truss and serve it on an AzureML online endpoint

This notebook walks through packaging `Qwen/Qwen3.5-0.8B` with **Truss** and serving it on an **AzureML managed online endpoint**, end to end, on a single GPU box. Everything it needs — the Truss `config.yaml`, the deployment template, and the deployment spec — is written inline as we go, so you can read it top to bottom without opening any other files.

## What we'll do
- Package the model with Truss's `docker_server` backend, which wraps the official `vllm/vllm-openai` image.
- Build the container here, run it on the GPU, and check it answers OpenAI-style `POST /v1/chat/completions`.
- Register the model in an AzureML registry, publish the image as an environment, and describe the serving contract with a Deployment Template.
- Deploy to a managed online endpoint and call it with the OpenAI SDK.

## How the pieces line up
Truss's `config.yaml` declares how the container serves. A Deployment Template carries the same information to AzureML, field for field:

| Truss `config.yaml` (`docker_server`) | AzureML Deployment Template |
|---|---|
| `server_port: 8000` | `scoring_port: 8000` + probe `port` |
| `predict_endpoint: /v1/chat/completions` | `scoring_path` |
| `readiness_endpoint: /health` | `readiness_probe.path` |
| `liveness_endpoint: /health` | `liveness_probe.path` |
| model weights | `model_mount_path` + a registered model |

The weights are provided at runtime by mounting the registered model at `/opt/ml/model`, and vLLM loads them from that path.

## Where to run this
On the `truss-build-a100` compute instance (`Standard_NC24ads_A100_v4`, A100 80 GB): it has both Docker and a GPU, which the build-and-test loop needs.

## Before you start
- Run this on the `truss-build-a100` compute instance (Docker + A100 GPU).
- `az` is already signed in — AzureML compute instances authenticate as you. Step 1 verifies it.
- The subscription, resource group, workspace, and registry are read from `~/.azureml/config.json` (or environment variables) in Step 2, so no identifiers are written into this notebook.

## Step 1 — Check the machine (GPU, Docker, disk, az)
A quick preflight so we fail fast if something is missing. One thing worth calling out: these boxes sometimes have both the older `azure-cli-ml` (v1) and the current `ml` (v2) CLI extensions installed. When both are present, `az ml ...` resolves to v1 and the v2 commands used later break — so we remove v1 if we find it.

In [1]:
import subprocess

def run(cmd):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, text=True).returncode

print("=== GPU ===")
run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo 'No GPU visible - are you on the A100 instance?'")

print("\n=== Docker ===")
run("docker version --format 'client={{.Client.Version}} server={{.Server.Version}}' || echo 'Docker not available'")

print("\n=== Docker can reach the GPU (pulls a small CUDA base once) ===")
run("docker run --rm --gpus all nvidia/cuda:12.4.0-base-ubuntu22.04 nvidia-smi -L || echo 'GPU-in-Docker check failed'")

print("\n=== Free disk (the vLLM image needs ~20-30 GB) ===")
run("df -h / /mnt 2>/dev/null | awk 'NR==1 || /\\/mnt|\\/$/'")

print("\n=== az sign-in ===")
run("az account show --query '{subscription:id, user:user.name}' -o yaml || echo 'Run: az login'")

print("\n=== AzureML CLI extension (want v2 'ml', not v1 'azure-cli-ml') ===")
exts = subprocess.run("az extension list --query \"[].name\" -o tsv", shell=True, capture_output=True, text=True).stdout
if "azure-cli-ml" in exts:
    print("Found azure-cli-ml (v1) - removing it so 'az ml' uses v2...")
    run("az extension remove -n azure-cli-ml")
run("az ml -h > /dev/null 2>&1 && echo 'az ml (v2) is working' || echo 'az ml v2 not working - install with: az extension add -n ml'")

=== GPU ===
$ nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo 'No GPU visible - are you on the A100 instance?'
NVIDIA A100 80GB PCIe, 81920 MiB, 580.159.03

=== Docker ===
$ docker version --format 'client={{.Client.Version}} server={{.Server.Version}}' || echo 'Docker not available'
client=24.0.9-1 server=24.0.9-1

=== Docker can reach the GPU (pulls a small CUDA base once) ===
$ docker run --rm --gpus all nvidia/cuda:12.4.0-base-ubuntu22.04 nvidia-smi -L || echo 'GPU-in-Docker check failed'
GPU 0: NVIDIA A100 80GB PCIe (UUID: GPU-4459a57e-f8fd-80eb-f65b-3b162096b8ca)

=== Free disk (the vLLM image needs ~20-30 GB) ===
$ df -h / /mnt 2>/dev/null | awk 'NR==1 || /\/mnt|\/$/'
Filesystem      Size  Used Avail Use% Mounted on
/dev/root       119G   95G   24G  81% /
/dev/sdb1        63G   32G   28G  54% /mnt

=== az sign-in ===
$ az account show --query '{subscription:id, user:user.name}' -o yaml || echo 'Run: az login'
subscription: 75703df0-38f9-4e2e

0

## Step 2 — Settings
We read the subscription, resource group, workspace, and registry from `~/.azureml/config.json` (environment variables override, if set). The asset names below are fixed for this walkthrough, and everything we create is versioned **10** so it's easy to tell these runs apart from earlier ones.

In [2]:
import os, json, glob, subprocess
from urllib.parse import urlparse

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

# Workspace/registry settings come from ~/.azureml/config.json (env vars win if set)
cfg = {}
for p in ["/home/azureuser/.azureml/config.json", os.path.expanduser("~/.azureml/config.json")]:
    if os.path.exists(p):
        cfg = json.load(open(p)); break

SUBSCRIPTION_ID = os.environ.get("SUBSCRIPTION_ID")  or cfg.get("subscription_id") or sh("az account show --query id -o tsv")
RESOURCE_GROUP  = os.environ.get("RESOURCE_GROUP")   or cfg.get("resource_group", "")
WORKSPACE       = os.environ.get("AZUREML_WORKSPACE") or cfg.get("workspace_name", "")
REGISTRY        = os.environ.get("AZUREML_REGISTRY")  or cfg.get("registry_name", "")

subprocess.run(f"az account set --subscription {SUBSCRIPTION_ID}", shell=True)

# Source model on Hugging Face
HF_MODEL_ID = "Qwen/Qwen3.5-0.8B"

# Names + versions for everything this notebook creates (version 10 = from this notebook)
VERSION       = "10"
MODEL_NAME    = "truss-vllm-qwen35"
ENV_NAME      = "truss-vllm-server"
DT_NAME       = "truss-vllm-qwen35-tp1"
ENDPOINT_NAME = "truss-vllm-qwen35-a100"
DEPLOY_NAME   = "truss-vllm-dep"
IMG           = f"{ENV_NAME}:{VERSION}"          # local docker tag
INSTANCE_TYPE = "Standard_NC24ads_A100_v4"

# Working directories (kept outside any git repo)
WORK      = os.path.expanduser("~/truss-vllm-run")
TRUSS_DIR = os.path.join(WORK, "truss-model")     # we write config.yaml here
BUILD_CTX = os.path.join(WORK, "build-context")   # truss generates the image context here
WEIGHTS   = os.path.join(WORK, "weights")         # HF weights land here
os.makedirs(WORK, exist_ok=True)

for k in ["SUBSCRIPTION_ID", "RESOURCE_GROUP", "WORKSPACE", "REGISTRY", "HF_MODEL_ID", "VERSION"]:
    print(f"{k:16}= {globals()[k] or '(missing - set it in ~/.azureml/config.json or an env var)'}")
assert SUBSCRIPTION_ID and RESOURCE_GROUP and WORKSPACE and REGISTRY, "Fill in the missing setting above."
print("\nSettings ready.")

SUBSCRIPTION_ID = 75703df0-38f9-4e2e-8328-45f6fc810286
RESOURCE_GROUP  = mabables-rg
WORKSPACE       = mabables-feb2026
REGISTRY        = mabables-reg-feb26
HF_MODEL_ID     = Qwen/Qwen3.5-0.8B
VERSION         = 10

Settings ready.


## Step 3 — Write the Truss package descriptor (`config.yaml`)
Truss turns a small `config.yaml` into a full container. We use the `docker_server` backend, which wraps an existing OpenAI-compatible server — here the official `vllm/vllm-openai` image. The fields that matter:

- `start_command` — how the server starts. We point vLLM at the mounted weights (`/opt/ml/model`) and serve on port 8000.
- `server_port: 8000` — the port the server listens on.
- `predict_endpoint: /v1/chat/completions` — the main inference route (OpenAI chat completions).
- `readiness_endpoint` / `liveness_endpoint: /health` — the health checks AzureML will probe.
- `base_image` — the image Truss builds on top of.

We write it to disk here so the rest of the notebook is self-contained.

In [3]:
os.makedirs(TRUSS_DIR, exist_ok=True)

config_yaml = """
model_name: qwen3-5-0-8b-vllm
python_version: py311

resources:
  accelerator: A100
  use_gpu: true

docker_server:
  start_command: vllm serve /opt/ml/model --served-model-name model --host 0.0.0.0 --port 8000 --tensor-parallel-size 1 --gpu-memory-utilization 0.9 --max-model-len 8192 --max-num-seqs 256
  server_port: 8000
  predict_endpoint: /v1/chat/completions
  readiness_endpoint: /health
  liveness_endpoint: /health

base_image:
  image: vllm/vllm-openai:latest

environment_variables:
  VLLM_TENSOR_PARALLEL_SIZE: "1"
  VLLM_GPU_MEMORY_UTILIZATION: "0.9"
  VLLM_MAX_MODEL_LEN: "8192"
  VLLM_MAX_NUM_SEQS: "256"
  VLLM_SERVED_MODEL_NAME: "model"
""".lstrip()

open(os.path.join(TRUSS_DIR, "config.yaml"), "w").write(config_yaml)
print(config_yaml)
print("Wrote", os.path.join(TRUSS_DIR, "config.yaml"))

model_name: qwen3-5-0-8b-vllm
python_version: py311

resources:
  accelerator: A100
  use_gpu: true

docker_server:
  start_command: vllm serve /opt/ml/model --served-model-name model --host 0.0.0.0 --port 8000 --tensor-parallel-size 1 --gpu-memory-utilization 0.9 --max-model-len 8192 --max-num-seqs 256
  server_port: 8000
  predict_endpoint: /v1/chat/completions
  readiness_endpoint: /health
  liveness_endpoint: /health

base_image:
  image: vllm/vllm-openai:latest

environment_variables:
  VLLM_TENSOR_PARALLEL_SIZE: "1"
  VLLM_GPU_MEMORY_UTILIZATION: "0.9"
  VLLM_MAX_MODEL_LEN: "8192"
  VLLM_MAX_NUM_SEQS: "256"
  VLLM_SERVED_MODEL_NAME: "model"

Wrote /home/azureuser/truss-vllm-run/truss-model/config.yaml


## Step 4 — Generate the container build context
`truss image build-context` reads `config.yaml` and produces a build context — a `Dockerfile`, an nginx config, and a supervisord config — that wraps vLLM. We don't hand-write the Dockerfile; Truss generates it. When you run the next cell it prints the actual one; it looks like this sample:

```dockerfile
FROM vllm/vllm-openai:latest AS truss_server
USER root
ENV DEBIAN_FRONTEND=noninteractive

# Truss rewrites apt to a community mirror (see note below)
RUN sed -i 's|http://archive.ubuntu.com/ubuntu/|mirror://mirrors.ubuntu.com/US.txt|g' /etc/apt/sources.list ...

# add nginx (reverse proxy) + a small control-plane venv on top of vLLM
RUN apt-get update -y && apt-get install -y --no-install-recommends curl nginx ...
RUN uv venv /docker_server/.venv --python 3.14 ...

COPY ./proxy.conf /etc/nginx/conf.d/proxy.conf
COPY ./supervisord.conf /etc/supervisor/supervisord.conf
COPY ./config.yaml /app/config.yaml
ENTRYPOINT ["/docker_server/.venv/bin/supervisord", "-c", "/etc/supervisor/supervisord.conf"]
```

What each part does:
- `FROM vllm/vllm-openai:latest` — the base image with vLLM already installed.
- The apt-mirror line — Truss rewrites apt sources to a community mirror. On a normal machine it resolves fine; inside some restricted build networks it can 404 on `apt-get install nginx`. If that happens there's a one-line fallback right after the build step.
- nginx + supervisord — Truss runs vLLM under supervisord and puts an nginx proxy in front of it. On AzureML we talk to vLLM on port 8000 directly, so nginx just sits idle.
- `COPY config.yaml` + `ENTRYPOINT supervisord` — the container boots supervisord, which launches the `start_command` from `config.yaml`.

A file like this is generated fresh every time you run Truss against our `config.yaml`.

In [4]:
import sys
subprocess.run(f'"{sys.executable}" -m pip install -q "truss==0.18.20"', shell=True, check=True)
truss_bin = os.path.join(os.path.dirname(sys.executable), "truss")

subprocess.run(f"rm -rf {BUILD_CTX}", shell=True)
subprocess.run(f'"{truss_bin}" image build-context {BUILD_CTX} {TRUSS_DIR} --non-interactive', shell=True, check=True)

print("=== Generated Dockerfile ===\n")
print(open(os.path.join(BUILD_CTX, "Dockerfile")).read())


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /home/azureuser/cloudfiles/code/.venv/bin/python -m pip install --upgrade pip
/home/azureuser/cloudfiles/code/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


=== Generated Dockerfile ===

ARG PYVERSION=py311
ARG HOME
FROM vllm/vllm-openai:latest AS truss_server

ENV PYTHON_EXECUTABLE="python3"

USER root

ENV HOME=${HOME:-/root}

ENV APP_HOME=/app
RUN mkdir -p ${APP_HOME} /control

RUN useradd -u 60000 -ms /bin/bash app

ENV DEBIAN_FRONTEND=noninteractive

RUN grep -w 'ID=debian\|ID_LIKE=debian' /etc/os-release || { echo "ERROR: Supplied base image is not a debian image"; exit 1; }
RUN $(which python3) -c "import sys; \
    sys.exit(0) \
    if sys.version_info.major == 3 \
    and sys.version_info.minor >= 9 \
    and sys.version_info.minor <= 14 \
    else sys.exit(1)" \
    || { echo "ERROR: Supplied base image does not have 3.9 <= python <= 3.14"; exit 1; }

RUN if [ -f /etc/apt/sources.list ]; then \
    sed -i.bak 's|http://archive.ubuntu.com/ubuntu/|mirror://mirrors.ubuntu.com/US.txt|g' /etc/apt/sources.list; \
fi
RUN if [ -f /etc/apt/sources.list.d/ubuntu.sources ]; then \
    sed -i.bak 's|http://archive.ubuntu.com/ubuntu/|mirror:/

## Step 5 — Build the image
We build on this box (native amd64, and the later push to the registry stays inside Azure). If `apt-get install nginx` fails with a mirror 404, re-run this cell (the mirror rotates on each try); if it keeps failing, run the fallback cell once and re-run this one.

In [5]:
rc = subprocess.run(f"cd {BUILD_CTX} && docker build -t {IMG} .", shell=True).returncode
print("\nBuild OK" if rc == 0 else "\nBuild failed - try the fallback cell below, then re-run this cell")

#0 building with "default" instance using docker driver

#1 [internal] load .dockerignore
#1 transferring context: 2B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 2.97kB done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/vllm/vllm-openai:latest
#3 DONE 0.2s

#4 [ 1/20] FROM docker.io/vllm/vllm-openai:latest@sha256:e4f88a835143cd22aee2397a26ec6bb80b3a4a6fe0c882bcbc63822904766089
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 5.93kB done
#5 DONE 0.0s

#6 [ 6/20] RUN if [ -f /etc/apt/sources.list ]; then     sed -i.bak 's|http://archive.ubuntu.com/ubuntu/|mirror://mirrors.ubuntu.com/US.txt|g' /etc/apt/sources.list; fi
#6 CACHED

#7 [ 7/20] RUN if [ -f /etc/apt/sources.list.d/ubuntu.sources ]; then     sed -i.bak 's|http://archive.ubuntu.com/ubuntu/|mirror://mirrors.ubuntu.com/US.txt|g' /etc/apt/sources.list.d/ubuntu.sources; fi
#7 CACHED

#8 [ 8/20] RUN command -v curl >/dev/null 2>&1 || (apt update &


Build OK


#25 exporting layers 6.2s done
#25 writing image sha256:c9b1ee0e04e99073567f90a8c231369d7e187722626597321b8cf9acc7816252 done
#25 naming to docker.io/library/truss-vllm-server:10 done
#25 DONE 6.2s


In [ ]:
# Fallback: run this ONLY if Step 5 failed on 'apt-get install nginx', then re-run Step 5.
# It points apt back at the canonical Ubuntu archive - only changes where apt downloads from.
# dfp = os.path.join(BUILD_CTX, "Dockerfile")
# s = open(dfp).read().replace("mirror://mirrors.ubuntu.com/US.txt", "http://archive.ubuntu.com/ubuntu/")
# open(dfp, "w").write(s)
# print("Reverted apt mirror. Re-run Step 5.")

## Step 6 — Download the model weights from Hugging Face
We pull `Qwen/Qwen3.5-0.8B` once and use the same copy twice: to mount into the local container for a test run (Step 7), and to upload when we register the model (Step 8).

In [6]:
import sys, importlib
subprocess.run(f'"{sys.executable}" -m pip install -q huggingface_hub', shell=True, check=True)
importlib.invalidate_caches()  # let the kernel see the just-installed package
from huggingface_hub import snapshot_download

subprocess.run(f"rm -rf {WEIGHTS} && mkdir -p {WEIGHTS}", shell=True)
snapshot_download(repo_id=HF_MODEL_ID, local_dir=WEIGHTS)

# vLLM expects config.json at the root of the model directory
assert os.path.exists(os.path.join(WEIGHTS, "config.json")), "config.json not found in the download"
print("Weights downloaded to:", WEIGHTS)
subprocess.run(f"ls -la {WEIGHTS} | head -20", shell=True)


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /home/azureuser/cloudfiles/code/.venv/bin/python -m pip install --upgrade pip
/home/azureuser/cloudfiles/code/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 13 files: 100%|██████████| 13/13 [00:17<00:00,  1.34s/it]

Weights downloaded to: /home/azureuser/truss-vllm-run/weights
total 1728540
drwxr-xr-x 3 azureuser azureuser       4096 Jul 19 03:47 .
drwxr-xr-x 5 azureuser azureuser       4096 Jul 19 03:47 ..
drwxr-xr-x 3 azureuser azureuser       4096 Jul 19 03:47 .cache
-rw-r--r-- 1 azureuser azureuser       1570 Jul 19 03:47 .gitattributes
-rw-r--r-- 1 azureuser azureuser      11544 Jul 19 03:47 LICENSE
-rw-r--r-- 1 azureuser azureuser      61705 Jul 19 03:47 README.md
-rw-r--r-- 1 azureuser azureuser       7755 Jul 19 03:47 chat_template.jinja
-rw-r--r-- 1 azureuser azureuser       2907 Jul 19 03:47 config.json
-rw-r--r-- 1 azureuser azureuser    3353259 Jul 19 03:47 merges.txt
-rw-r--r-- 1 azureuser azureuser 1746942600 Jul 19 03:47 model.safetensors-00001-of-00001.safetensors
-rw-r--r-- 1 azureuser azureuser      50900 Jul 19 03:47 model.safetensors.index.json
-rw-r--r-- 1 azureuser azureuser        390 Jul 19 03:47 preprocessor_config.json
-rw-r--r-- 1 azureuser azureuser   12807982 Jul 19 03

CompletedProcess(args='ls -la /home/azureuser/truss-vllm-run/weights | head -20', returncode=0)

## Step 7 — Run the container on the GPU and check it
Start the image we just built, mount the weights at `/opt/ml/model` (the same path AzureML will use), wait for `/health` to go green, and send a chat completion. We stop the container afterwards to free the GPU before pushing the image.

In [7]:
import time, urllib.request

subprocess.run("docker rm -f truss-vllm-test 2>/dev/null", shell=True)
subprocess.run(
    f'docker run -d --name truss-vllm-test --gpus all '
    f'-v "{WEIGHTS}":/opt/ml/model -p 8000:8000 {IMG}',
    shell=True, check=True)

def _status(url):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.status
    except Exception:
        return None

print("Waiting for vLLM to load the model...")
ready = False
for i in range(72):  # up to ~6 minutes
    if _status("http://localhost:8000/health") == 200:
        ready = True; print(f"/health -> 200 after ~{i*5}s"); break
    time.sleep(5)
if not ready:
    subprocess.run("docker logs --tail 80 truss-vllm-test", shell=True)
    raise SystemExit("vLLM did not become healthy")

req = json.dumps({
    "model": "model",
    "messages": [{"role": "user", "content": "What is the capital of France?"}],
    "max_tokens": 30, "temperature": 0.1,
}).encode()
r = urllib.request.Request("http://localhost:8000/v1/chat/completions",
                           data=req, headers={"Content-Type": "application/json"})
with urllib.request.urlopen(r, timeout=90) as resp:
    body = json.load(resp)

print("\n=== POST /v1/chat/completions ===")
print(json.dumps(body, indent=2)[:800])
assert body.get("object") == "chat.completion"
print("\nWorks. Stopping the local container.")
subprocess.run("docker rm -f truss-vllm-test 2>/dev/null", shell=True)

7de46c5c8cd4ce53a5c01cc1c47668daf7de11df86d3e13e1aa2a06de1161b2b
Waiting for vLLM to load the model...
/health -> 200 after ~195s

=== POST /v1/chat/completions ===
{
  "id": "chatcmpl-ab99c9cfc91b58ad",
  "object": "chat.completion",
  "created": 1784433090,
  "model": "model",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "The capital of France is **Paris**.\n\nLocated in the heart of the country, Paris is the largest city in France and serves as the seat of",
        "refusal": null,
        "annotations": null,
        "audio": null,
        "function_call": null,
        "reasoning": null
      },
      "logprobs": null,
      "finish_reason": "length",
      "stop_reason": null,
      "token_ids": null,
      "routed_experts": null
    }
  ],
  "service_tier": null,
  "system_fingerprint": "vllm-0.25.1-a9e4f5d7",
  "usage": {
    "prompt_tokens": 19,
    "total_tokens": 49,
    "complet

Works. Stopping the local contain

CompletedProcess(args='docker rm -f truss-vllm-test 2>/dev/null', returncode=0)

## Step 8 — Register the model in the registry
The deployment mounts the model at runtime, and for that the registered model needs a small manifest that a plain `az ml model create` from a folder doesn't produce. We use the registry's upload flow: ask for a temporary blob location, upload the weights with `azcopy`, then create the model version pointing at that blob with the manifest flag set.

In [8]:
import urllib.request

ARM  = "https://management.azure.com"
API  = "2025-04-01-preview"
BASE = f"{ARM}/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.MachineLearningServices/registries/{REGISTRY}"

def _token():
    return sh("az account get-access-token --query accessToken -o tsv")

def rest(method, url, body=None):
    data = json.dumps(body).encode() if body is not None else None
    rq = urllib.request.Request(url, data=data, method=method)
    rq.add_header("Authorization", f"Bearer {_token()}")
    rq.add_header("Content-Type", "application/json")
    with urllib.request.urlopen(rq, timeout=120) as resp:
        raw = resp.read().decode()
        return json.loads(raw) if raw else {}

# 1) ask for a temporary blob location for this model version
pending = rest("POST",
               f"{BASE}/models/{MODEL_NAME}/versions/{VERSION}/startPendingUpload?api-version={API}",
               {"pendingUploadType": "TemporaryBlobReference"})
sas_uri  = pending["blobReferenceForConsumption"]["credential"]["sasUri"]
blob_uri = pending["blobReferenceForConsumption"]["blobUri"]
print("Got a temporary upload location.")

# 2) upload the weights
subprocess.run(
    f'azcopy copy "{WEIGHTS}/*" "{sas_uri}" --recursive --put-md5 --log-level WARNING --output-level essential',
    shell=True, check=True)

# 3) create the model version pointing at the blob, with the manifest flag
rest("PUT", f"{BASE}/models/{MODEL_NAME}/versions/{VERSION}?api-version={API}", {
    "properties": {
        "description": "Qwen/Qwen3.5-0.8B weights, mounted at runtime",
        "modelType": "custom_model",
        "modelUri": blob_uri,
        "properties": {"aotManifest": "True"},
        "tags": {"framework": "truss", "hf_model_id": HF_MODEL_ID},
    }
})

# 4) confirm it registered with a manifest
info = rest("GET", f"{BASE}/models/{MODEL_NAME}/versions/{VERSION}?api-version={API}")
print(f"Registered {MODEL_NAME}:{VERSION}")
print("manifest:", info.get("properties", {}).get("properties"))

Got a temporary upload location.

Job bf4f4bb6-3458-5b45-6955-d6ef90e36784 has started
Log file is located at: /home/azureuser/.azcopy/bf4f4bb6-3458-5b45-6955-d6ef90e36784.log




Job bf4f4bb6-3458-5b45-6955-d6ef90e36784 summary
Elapsed Time (Minutes): 0.0667
Number of File Transfers: 42
Number of Folder Property Transfers: 0
Number of Symlink Transfers: 0
Total Number of Transfers: 42
Number of File Transfers Completed: 42
Number of Folder Transfers Completed: 0
Number of File Transfers Failed: 0
Number of Folder Transfers Failed: 0
Number of File Transfers Skipped: 0
Number of Folder Transfers Skipped: 0
Number of Symbolic Links Skipped: 0
Number of Hardlinks Converted: 0
Number of Hardlinks Skipped: 0
Number of Special Files Skipped: 0
Total Number of Bytes Transferred: 1769983830
Final Job Status: Completed

Registered truss-vllm-qwen35:10
manifest: {'modelManifestPathOrUri': '/mabables-r-487ca3f6-b979-5d2c-a1dd-399c92d3b4dc/manifest.base.json'}


## Step 9 — Push the image to the workspace registry (ACR)
Tag the image for the workspace's container registry and push it. This stays inside Azure, so it's quick.

In [9]:
ACR_ID   = sh(f"az ml workspace show --name {WORKSPACE} --resource-group {RESOURCE_GROUP} --query container_registry -o tsv")
ACR_NAME = ACR_ID.split("/")[-1]
ACR_IMG  = f"{ACR_NAME}.azurecr.io/{ENV_NAME}:{VERSION}"
print("workspace ACR:", ACR_NAME)

subprocess.run(f"az acr login --name {ACR_NAME}", shell=True, check=True)
subprocess.run(f"docker tag {IMG} {ACR_IMG}", shell=True, check=True)
subprocess.run(f"docker push {ACR_IMG}", shell=True, check=True)
print("pushed:", ACR_IMG)

workspace ACR: c47421369908446eb3b9043f0033e4dc
Login Succeeded
The push refers to repository [c47421369908446eb3b9043f0033e4dc.azurecr.io/truss-vllm-server]
9563f556885d: Preparing
5f70bf18a086: Preparing
4877e7cbf52e: Preparing
22f28249a086: Preparing
727d1bb91377: Preparing
2cd8896a2cb5: Preparing
4c3d9d5af3eb: Preparing
3db492b5e9be: Preparing
4348cea7674e: Preparing
25c6d19d617d: Preparing
cc5ae5993323: Preparing
5f70bf18a086: Preparing
5f70bf18a086: Preparing
5f70bf18a086: Preparing
da5815e2a19e: Preparing
e44da520aac4: Preparing
5f70bf18a086: Preparing
70d6afb510c3: Preparing
19d5a79dd471: Preparing
e7983e951bef: Preparing
dd0040d0694a: Preparing
18c598861fc3: Preparing
3fd833301c44: Preparing
8dd6ef296e5d: Preparing
a0676a46c1b2: Preparing
3efba386d495: Preparing
b8365a3ac6c4: Preparing
5f70bf18a086: Preparing
e04ae5d70b43: Preparing
467ae0a4780f: Preparing
4c7a720583f7: Preparing
d6a87a2f5020: Preparing
20edd76d0016: Preparing
ee54c51ad45e: Preparing
f1a5a58b1fc8: Preparing
40

## Step 10 — Create the environment and copy it into the registry
Register an AzureML environment that references the pushed image (no build step), then share it into the registry so a Deployment Template can use it.

In [10]:
env_yaml = f"""$schema: https://azuremlschemas.azureedge.net/latest/environment.schema.json
name: {ENV_NAME}
version: "{VERSION}"
image: {ACR_IMG}
description: "Truss docker_server + vLLM (Qwen3.5-0.8B)"
"""
open(f"{WORK}/env.yml", "w").write(env_yaml)
subprocess.run(
    f"az ml environment create --file {WORK}/env.yml "
    f"--resource-group {RESOURCE_GROUP} --workspace-name {WORKSPACE}",
    shell=True, check=True)
print(f"workspace environment created: {ENV_NAME}:{VERSION}")

subprocess.run(
    f"az ml environment share --name {ENV_NAME} --version {VERSION} "
    f"--resource-group {RESOURCE_GROUP} --workspace-name {WORKSPACE} "
    f"--registry-name {REGISTRY} --share-with-name {ENV_NAME} --share-with-version {VERSION}",
    shell=True, check=True)
print(f"shared to registry: {REGISTRY}")

{
  "creation_context": {
    "created_at": "2026-07-19T03:52:56.097570+00:00",
    "created_by": "Manoj Bableshwar",
    "created_by_type": "User",
    "last_modified_at": "2026-07-19T03:52:56.097570+00:00",
    "last_modified_by": "Manoj Bableshwar",
    "last_modified_by_type": "User"
  },
  "description": "Truss docker_server + vLLM (Qwen3.5-0.8B)",
  "id": "azureml:/subscriptions/75703df0-38f9-4e2e-8328-45f6fc810286/resourceGroups/mabables-rg/providers/Microsoft.MachineLearningServices/workspaces/mabables-feb2026/environments/truss-vllm-server/versions/10",
  "image": "c47421369908446eb3b9043f0033e4dc.azurecr.io/truss-vllm-server:10",
  "name": "truss-vllm-server",
  "os_type": "linux",
  "resourceGroup": "mabables-rg",
  "tags": {},
  "version": "10"
}
workspace environment created: truss-vllm-server:10


Method share: This is an experimental method, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


{
  "creation_context": {
    "created_at": "2026-07-19T03:53:05.861416+00:00",
    "created_by": "Manoj Bableshwar",
    "created_by_type": "User",
    "last_modified_at": "2026-07-19T03:53:05.861416+00:00",
    "last_modified_by": "Manoj Bableshwar",
    "last_modified_by_type": "User"
  },
  "description": "Truss docker_server + vLLM (Qwen3.5-0.8B)",
  "id": "azureml://registries/mabables-reg-feb26/environments/truss-vllm-server/versions/10",
  "image": "c47421369908446eb3b9043f0033e4dc.azurecr.io/truss-vllm-server:10",
  "name": "truss-vllm-server",
  "os_type": "linux",
  "tags": {},
  "version": "10"
}
shared to registry: mabables-reg-feb26


## Step 11 — Describe the serving contract with a Deployment Template
The Deployment Template tells AzureML how to run the container: which image (the registry environment), the port to send scoring traffic to, the scoring path, the health probes, and where to mount the model. These mirror the `config.yaml` fields from Step 3.

We then point the registered model at this template. With that link in place, the deployment in Step 13 stays tiny — it just names the model and inherits the environment, port, path, probes, and mount path from the template.

In [14]:
dt_yaml = f"""$schema: https://azuremlschemas.azureedge.net/latest/deploymentTemplate.schema.json
name: {DT_NAME}
version: "{VERSION}"
description: "Truss + vLLM (Qwen3.5-0.8B)"
deployment_template_type: managed
environment: azureml://registries/{REGISTRY}/environments/{ENV_NAME}/versions/{VERSION}
default_instance_type: {INSTANCE_TYPE}
allowed_instance_types:
  - {INSTANCE_TYPE}
instance_count: 1
scoring_port: 8000
scoring_path: /v1/chat/completions
model_mount_path: /opt/ml/model
environment_variables:
  PYTHONUNBUFFERED: "1"
request_settings:
  request_timeout_ms: 90000
  max_concurrent_requests_per_instance: 256
liveness_probe:
  initial_delay: 600
  period: 10
  timeout: 10
  success_threshold: 1
  scheme: http
  method: GET
  path: /health
  port: 8000
readiness_probe:
  initial_delay: 600
  period: 10
  timeout: 10
  success_threshold: 1
  scheme: http
  method: GET
  path: /health
  port: 8000
"""
open(f"{WORK}/deployment-template.yml", "w").write(dt_yaml)
print(dt_yaml)

# Create the template (tolerate re-runs where this version already exists)
subprocess.run(
    f"az ml deployment-template create --file {WORK}/deployment-template.yml "
    f"--registry-name {REGISTRY} --version {VERSION}",
    shell=True)
print(f"deployment template ready: {DT_NAME}:{VERSION}")

# Point the model at this template so the deployment inherits its environment,
# port, path, probes, and mount path (a model-registry data-plane PATCH).
import urllib.request, urllib.error

REGISTRY_LOCATION = sh(f"az ml registry show --name {REGISTRY} --query location -o tsv")
MFE = (f"https://{REGISTRY_LOCATION}.api.azureml.ms/modelregistry/v1.0/subscriptions/{SUBSCRIPTION_ID}"
       f"/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.MachineLearningServices/registries/{REGISTRY}"
       f"/models/{MODEL_NAME}:{VERSION}")
DT_REF = f"azureml://registries/{REGISTRY}/deploymentTemplates/{DT_NAME}/versions/{VERSION}"

def _patch(body):
    tok = sh("az account get-access-token --query accessToken -o tsv")
    rq = urllib.request.Request(MFE, data=json.dumps(body).encode(), method="PATCH")
    rq.add_header("Authorization", f"Bearer {tok}")
    rq.add_header("Content-Type", "application/json")
    try:
        with urllib.request.urlopen(rq, timeout=60) as resp:
            return resp.status
    except urllib.error.HTTPError as e:
        return e.code

# remove any prior link, then set the default + allowed templates
_patch([{"op": "remove", "path": "/defaultDeploymentTemplate"}])
code = _patch([{"op": "add", "path": "/defaultDeploymentTemplate", "value": {"assetId": DT_REF}}])
_patch([{"op": "add", "path": "/allowedDeploymentTemplates",
         "value": [{"assetId": f"azureml://registries/{REGISTRY}/deploymentTemplates/{DT_NAME}/labels/latest"}]}])
print(f"linked model {MODEL_NAME}:{VERSION} -> {DT_NAME}:{VERSION} (HTTP {code})")

$schema: https://azuremlschemas.azureedge.net/latest/deploymentTemplate.schema.json
name: truss-vllm-qwen35-tp1
version: "10"
description: "Truss + vLLM (Qwen3.5-0.8B)"
deployment_template_type: managed
environment: azureml://registries/mabables-reg-feb26/environments/truss-vllm-server/versions/10
default_instance_type: Standard_NC24ads_A100_v4
allowed_instance_types:
  - Standard_NC24ads_A100_v4
instance_count: 1
scoring_port: 8000
scoring_path: /v1/chat/completions
model_mount_path: /opt/ml/model
environment_variables:
  PYTHONUNBUFFERED: "1"
request_settings:
  request_timeout_ms: 90000
  max_concurrent_requests_per_instance: 256
liveness_probe:
  initial_delay: 600
  period: 10
  timeout: 10
  success_threshold: 1
  scheme: http
  method: GET
  path: /health
  port: 8000
readiness_probe:
  initial_delay: 600
  period: 10
  timeout: 10
  success_threshold: 1
  scheme: http
  method: GET
  path: /health
  port: 8000



Method load_deployment_template: This is an experimental method, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class DeploymentTemplateSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class RequestSettingsSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class OnlineRequestSettings: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProbeSettingsSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProbeSettings: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class DeploymentTemplate: This is an experimental class, and may c

{
  "allowedInstanceTypes": [
    "Standard_NC24ads_A100_v4"
  ],
  "defaultInstanceType": "Standard_NC24ads_A100_v4",
  "deploymentTemplateType": "managed",
  "description": "Truss + vLLM (Qwen3.5-0.8B)",
  "environmentId": "azureml://registries/mabables-reg-feb26/environments/truss-vllm-server/versions/10",
  "environmentVariables": {
    "PYTHONUNBUFFERED": "1"
  },
  "instanceCount": 1,
  "livenessProbe": {
    "httpMethod": "GET",
    "initialDelay": "PT10M",
    "path": "/health",
    "period": "PT10S",
    "port": 8000,
    "scheme": "http",
    "successThreshold": 1,
    "timeout": "PT10S"
  },
  "modelMountPath": "/opt/ml/model",
  "name": "truss-vllm-qwen35-tp1",
  "readinessProbe": {
    "httpMethod": "GET",
    "initialDelay": "PT10M",
    "path": "/health",
    "period": "PT10S",
    "port": 8000,
    "scheme": "http",
    "successThreshold": 1,
    "timeout": "PT10S"
  },
  "requestSettings": {
    "maxConcurrentRequestsPerInstance": 256,
    "requestTimeout": "PT1M30S"
 

## Step 12 — Create the online endpoint
The endpoint is the front door: a URL and an auth key. We create it with key auth.

In [12]:
ep_yaml = f"""$schema: https://azuremlschemas.azureedge.net/latest/managedOnlineEndpoint.schema.json
name: {ENDPOINT_NAME}
auth_mode: key
description: "Truss + vLLM (Qwen3.5-0.8B)"
"""
open(f"{WORK}/endpoint.yml", "w").write(ep_yaml)
subprocess.run(
    f"az ml online-endpoint create --file {WORK}/endpoint.yml "
    f"--resource-group {RESOURCE_GROUP} --workspace-name {WORKSPACE}",
    shell=True, check=True)
print(f"endpoint created: {ENDPOINT_NAME}")

{
  "auth_mode": "key",
  "description": "Truss + vLLM (Qwen3.5-0.8B)",
  "id": "/subscriptions/75703df0-38f9-4e2e-8328-45f6fc810286/resourceGroups/mabables-rg/providers/Microsoft.MachineLearningServices/workspaces/mabables-feb2026/onlineEndpoints/truss-vllm-qwen35-a100",
  "identity": {
    "principal_id": "1f96e276-4380-4ef2-9ea0-2abe0b75b23d",
    "tenant_id": "7f292395-a08f-4cc0-b3d0-a400b023b0d2",
    "type": "system_assigned"
  },
  "kind": "Managed",
  "location": "eastus2",
  "mirror_traffic": {},
  "name": "truss-vllm-qwen35-a100",
  "openapi_uri": "https://truss-vllm-qwen35-a100.eastus2.inference.ml.azure.com/swagger.json",
  "properties": {
    "AzureAsyncOperationUri": "https://management.azure.com/subscriptions/75703df0-38f9-4e2e-8328-45f6fc810286/providers/Microsoft.MachineLearningServices/locations/eastus2/mfeOperationsStatus/oeidp:c4742136-9908-446e-b3b9-043f0033e4dc:f4600dc8-b84d-416a-aa5a-a9c532b4245f?api-version=2022-02-01-preview",
    "azureml.onlineendpointid": "/

## Step 13 — Deploy to the endpoint
The deployment ties together the registered model (mounted at `/opt/ml/model`) and the Deployment Template (image + serving contract). Provisioning a GPU node, pulling the image, and loading the model takes roughly 15-30 minutes.

In [15]:
dep_yaml = f"""$schema: https://azuremlschemas.azureedge.net/latest/managedOnlineDeployment.schema.json
name: {DEPLOY_NAME}
endpoint_name: {ENDPOINT_NAME}
model: azureml://registries/{REGISTRY}/models/{MODEL_NAME}/versions/{VERSION}
instance_type: {INSTANCE_TYPE}
instance_count: 1
properties:
  azureml.deploymentTemplateOverride: "azureml://registries/{REGISTRY}/deploymenttemplates/{DT_NAME}/versions/{VERSION}"
"""
open(f"{WORK}/deployment.yml", "w").write(dep_yaml)
subprocess.run(
    f"az ml online-deployment create --file {WORK}/deployment.yml "
    f"--resource-group {RESOURCE_GROUP} --workspace-name {WORKSPACE} --all-traffic",
    shell=True, check=True)
print(f"deployment created and taking traffic: {DEPLOY_NAME}")

ActivityStarted, Workspace.Get
ActivityCompleted: Activity=Workspace.Get, HowEnded=Success, Duration=1830.78 [ms]
All traffic will be set to deployment truss-vllm-dep once it has been provisioned.
If you interrupt this command or it times out while waiting for the provisioning, you can try to set all the traffic to this deployment later once its has been provisioned.
Check: endpoint truss-vllm-qwen35-a100 exists

Model 'truss-vllm-qwen35' (version 10) from registry 'mabables-reg-feb26' has a default deployment template configured.
The deployment will use the default deployment template settings, and some deployment parameters may be ignored.
Default deployment template: azureml://registries/mabables-reg-feb26/deploymentTemplates/truss-vllm-qwen35-tp1/versions/10


................................................................................................................................................................................................................................{
  "app_insights_enabled": false,
  "creation_context": {
    "created_at": "2026-07-19T03:59:23.584023+00:00",
    "created_by": "Manoj Bableshwar",
    "last_modified_at": "2026-07-19T03:59:23.584028+00:00"
  },
  "egress_public_network_access": "enabled",
  "endpoint_name": "truss-vllm-qwen35-a100",
  "environment_variables": {},
  "id": "/subscriptions/75703df0-38f9-4e2e-8328-45f6fc810286/resourceGroups/mabables-rg/providers/Microsoft.MachineLearningServices/workspaces/mabables-feb2026/onlineEndpoints/truss-vllm-qwen35-a100/deployments/truss-vllm-dep",
  "instance_count": 1,
  "instance_type": "Standard_NC24ads_A100_v4",
  "model": "azureml://registries/mabables-reg-feb26/models/truss-vllm-qwen35/versions/10",
  "name": "truss-vllm-dep",
  "properties": {
    "

## Step 14 — Call the endpoint with the OpenAI SDK
Fetch the endpoint URL and key, point the OpenAI client at `{endpoint}/v1`, and send a chat completion. Because the scoring path is `/v1/chat/completions`, the endpoint's URL is the OpenAI chat-completions URL directly.

In [17]:
import sys, importlib
subprocess.run(f'"{sys.executable}" -m pip install -q openai', shell=True, check=True)
importlib.invalidate_caches()  # let the kernel see the just-installed package
from openai import OpenAI

scoring_uri = sh(f"az ml online-endpoint show -n {ENDPOINT_NAME} -g {RESOURCE_GROUP} -w {WORKSPACE} --query scoring_uri -o tsv")
key         = sh(f"az ml online-endpoint get-credentials -n {ENDPOINT_NAME} -g {RESOURCE_GROUP} -w {WORKSPACE} --query primaryKey -o tsv")
u = urlparse(scoring_uri)
base_url = f"{u.scheme}://{u.netloc}/v1"
print("scoring_uri:", scoring_uri)
print("base_url   :", base_url)

client = OpenAI(base_url=base_url, api_key=key)
resp = client.chat.completions.create(
    model="model",
    messages=[{"role": "user", "content": "Capital of France?"}],
    max_tokens=30,
)
print("\nobject :", resp.object)
print("model  :", resp.model)
print("answer :", resp.choices[0].message.content)
print("finish :", resp.choices[0].finish_reason)


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /home/azureuser/cloudfiles/code/.venv/bin/python -m pip install --upgrade pip


scoring_uri: https://truss-vllm-qwen35-a100.eastus2.inference.ml.azure.com/v1/chat/completions
base_url   : https://truss-vllm-qwen35-a100.eastus2.inference.ml.azure.com/v1

object : chat.completion
model  : model
answer : The capital city of France is **Paris**.

French territories and overseas departments were historically ceded from England but recently acquired by France from Spain, Puerto
finish : length


## Wrap-up
The endpoint is live and answering OpenAI `POST /v1/chat/completions`. What we created (all version 10):

- Model `truss-vllm-qwen35:10` — registry, weights mounted at runtime
- Environment `truss-vllm-server:10` — workspace and registry
- Deployment template `truss-vllm-qwen35-tp1:10`
- Endpoint `truss-vllm-qwen35-a100` with deployment `truss-vllm-dep`

When you're done, remove the endpoint and (optionally) stop this instance:

```bash
az ml online-endpoint delete -n truss-vllm-qwen35-a100 -g <resource-group> -w <workspace> --yes
az ml compute stop --name truss-build-a100 -g <resource-group> -w <workspace>
```